In [13]:
import pandas as pd
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import os
import numpy as np

In [14]:
df = pd.read_csv('../data/raw/final_merged_data.csv')
df.head(30)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,Participant
0,1.613036e+12,NaN,[LogManager] Application Started,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
1,1.613036e+12,NaN,[UnityRecordSection@10Hz],Recorder added : TransformRecorder:Player,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
2,1.613036e+12,NaN,[UnityRecordSection@10Hz],Recorder added : LogEventIDRecorder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
3,1.613036e+12,NaN,[ExperimentManager] Session changed to: S0_Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
4,1.613036e+12,1.0,[ S0_Exploration ] Pause ON,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
5,1.613036e+12,NaN,[ExperimentManager] SubjectID set: P10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
6,1.613036e+12,NaN,[ExperimentManager] Session changed to: S1_Sprint,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
7,1.613036e+12,2.0,[ S1_Sprint ] Pause ON,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
8,1.613036e+12,NaN,[ExperimentManager] Session changed to: S2_Exp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
9,1.613036e+12,3.0,[ S2_Experiment ] Pause ON,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10


In [15]:
# only keep V3 is [ S3_Memory ] Stimuli Placed or [ S3_Memory ] Memory Placement confidence
df = df[df['V3'].isin(['[ S3_Memory ] Stimuli Placed', '[ S3_Memory ] Memory Placement confidence', '[ S3_Memory ] Memory Quiz2 Response', '[ S3_Memory ] Memory Quiz2 confidence'])]
# Give adjacent indices to each stimulus placement and confidence rating
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,Participant
22,1.613036e+12,15.0,[ S3_Memory ] Stimuli Placed,food_2_e1_11_nr_o_v1_old,object,nr,Checkpoint : 11,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
23,1.613036e+12,16.0,[ S3_Memory ] Memory Placement confidence,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
25,1.613036e+12,18.0,[ S3_Memory ] Memory Quiz2 Response,two_weeks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
26,1.613036e+12,19.0,[ S3_Memory ] Memory Quiz2 confidence,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10
32,1.613036e+12,25.0,[ S3_Memory ] Stimuli Placed,drink_11_memo,object,ne,Checkpoint : 0,"Checkpoint Pos : (-131.4, 107.0, 40.1)","Stimuli Pos : (-187.6, 107.0, 1156.9)",Distance : 1118.311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,P10


In [16]:
placed = df[df['V3'] == '[ S3_Memory ] Stimuli Placed']
placed_confidence = df[df['V3'] == '[ S3_Memory ] Memory Placement confidence']
timed = df[df['V3'] == '[ S3_Memory ] Memory Quiz2 Response']
timed_confidence = df[df['V3'] == '[ S3_Memory ] Memory Quiz2 confidence']
# reset index
placed = placed.reset_index(drop=True)
placed_confidence = placed_confidence.reset_index(drop=True)
timed = timed.reset_index(drop=True)
timed_confidence = timed_confidence.reset_index(drop=True)

timed = timed.rename(columns={'V4': 'Timed'})[['Timed']]
placed_confidence = placed_confidence.rename(columns={'V4': 'Placed Confidence'})[['Placed Confidence']]
timed_confidence = timed_confidence.rename(columns={'V4': 'Timed Confidence'})[['Timed Confidence']]
print(timed.head())
# merge on index
merged = pd.merge(placed, placed_confidence, left_index=True, right_index=True)
merged = pd.merge(merged, timed, left_index=True, right_index=True)
merged = pd.merge(merged, timed_confidence, left_index=True, right_index=True)
merged.head()

       Timed
0  two_weeks
1      never
2  yesterday
3      never
4      never


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V14,V15,V16,V17,V18,V19,Participant,Placed Confidence,Timed,Timed Confidence
0,1.613036e+12,15.0,[ S3_Memory ] Stimuli Placed,food_2_e1_11_nr_o_v1_old,object,nr,Checkpoint : 11,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,...,NaN,NaN,NaN,NaN,NaN,NaN,P10,23,two_weeks,37
1,1.613036e+12,25.0,[ S3_Memory ] Stimuli Placed,drink_11_memo,object,ne,Checkpoint : 0,"Checkpoint Pos : (-131.4, 107.0, 40.1)","Stimuli Pos : (-187.6, 107.0, 1156.9)",Distance : 1118.311,...,NaN,NaN,NaN,NaN,NaN,NaN,P10,100,never,100
2,1.613037e+12,35.0,[ S3_Memory ] Stimuli Placed,wealth_6_e2_15_r_o_v1_old,object,re,Checkpoint : 15,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",Distance : 543.8705,...,NaN,NaN,NaN,NaN,NaN,NaN,P10,4,yesterday,46
3,1.613037e+12,45.0,[ S3_Memory ] Stimuli Placed,pirate_2_e3_11_nr_h_v1_old,human,nr,Checkpoint : 11,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",Distance : 754.4911,...,NaN,NaN,NaN,NaN,NaN,NaN,P10,78,never,78
4,1.613037e+12,55.0,[ S3_Memory ] Stimuli Placed,viking_8_e3_16_nr_h_v1_old,human,nr,Checkpoint : 16,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",Distance : 745.0938,...,NaN,NaN,NaN,NaN,NaN,NaN,P10,32,never,21


In [17]:
# Only keep relevant columns
df = merged[['Participant','V1', 'V4', 'V8', 'V9', 'V10', 'Placed Confidence', 'Timed', 'Timed Confidence']]
df.head()

,Participant,V1,V4,V8,V9,V10,Placed Confidence,Timed,Timed Confidence
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,23,two_weeks,37
1,P10,1.613036e+12,drink_11_memo,"Checkpoint Pos : (-131.4, 107.0, 40.1)","Stimuli Pos : (-187.6, 107.0, 1156.9)",Distance : 1118.311,100,never,100
2,P10,1.613037e+12,wealth_6_e2_15_r_o_v1_old,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",Distance : 543.8705,4,yesterday,46
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",Distance : 754.4911,78,never,78
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",Distance : 745.0938,32,never,21


In [18]:
# Only keep V4 valus that have old in it
df = df[df['V4'].str.contains('old')]

In [19]:
df[['X', 'Y', 'Z']] = df['V9'].str.extract(
    r'Stimuli Pos\s*:\s*\(\s*([-\d.]+),\s*([-\d.]+),\s*([-\d.]+)\s*\)'
).astype(float)
df.head()

,Participant,V1,V4,V8,V9,V10,Placed Confidence,Timed,Timed Confidence,X,Y,Z
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,23,two_weeks,37,-221.7,88.3,283.6
2,P10,1.613037e+12,wealth_6_e2_15_r_o_v1_old,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",Distance : 543.8705,4,yesterday,46,-116.3,63.4,272.2
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",Distance : 754.4911,78,never,78,-38.6,88.3,1182.9
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",Distance : 745.0938,32,never,21,-142.3,53.3,1275.2
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,"Checkpoint Pos : (-128.5, 90.6, 220.8)","Stimuli Pos : (-129.3, 90.6, 220.4)",Distance : 0.9522193,78,two_weeks,82,-129.3,90.6,220.4


In [20]:
df['Timed'].unique()

array(['two_weeks', 'yesterday', 'never', 'one_week'], dtype=object)

In [21]:
# change Timed column to numeric values
df['Timed'] = df['Timed'].map({'yesterday': 1, 'one_week': 7, 'two_weeks': 14, 'never': 100})
# rename column
df = df.rename(columns={'Timed': 'Timed (days)'})
df.head()

,Participant,V1,V4,V8,V9,V10,Placed Confidence,Timed (days),Timed Confidence,X,Y,Z
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,23,14,37,-221.7,88.3,283.6
2,P10,1.613037e+12,wealth_6_e2_15_r_o_v1_old,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",Distance : 543.8705,4,1,46,-116.3,63.4,272.2
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",Distance : 754.4911,78,100,78,-38.6,88.3,1182.9
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",Distance : 745.0938,32,100,21,-142.3,53.3,1275.2
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,"Checkpoint Pos : (-128.5, 90.6, 220.8)","Stimuli Pos : (-129.3, 90.6, 220.4)",Distance : 0.9522193,78,14,82,-129.3,90.6,220.4


In [22]:
# Add true time column 1 if e3 in V4, 7 if e2 in V4, 14 if e1 in V4
df['True Time (days)'] = df['V4'].map(lambda x: 1 if 'e3' in x else (7 if 'e2' in x else (14 if 'e1' in x else 100)))
df.head()

,Participant,V1,V4,V8,V9,V10,Placed Confidence,Timed (days),Timed Confidence,X,Y,Z,True Time (days)
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",Distance : 222.2831,23,14,37,-221.7,88.3,283.6,14
2,P10,1.613037e+12,wealth_6_e2_15_r_o_v1_old,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",Distance : 543.8705,4,1,46,-116.3,63.4,272.2,7
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",Distance : 754.4911,78,100,78,-38.6,88.3,1182.9,1
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",Distance : 745.0938,32,100,21,-142.3,53.3,1275.2,1
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,"Checkpoint Pos : (-128.5, 90.6, 220.8)","Stimuli Pos : (-129.3, 90.6, 220.4)",Distance : 0.9522193,78,14,82,-129.3,90.6,220.4,1


In [23]:
# remove 'Distance : ' from V10 and convert to float
df['Distance'] = df['V10'].str.replace('Distance :', '').astype(float)
df.drop('V10', axis=1, inplace=True)
df.head()

,Participant,V1,V4,V8,V9,Placed Confidence,Timed (days),Timed Confidence,X,Y,Z,True Time (days),Distance
0,P10,1.613036e+12,food_2_e1_11_nr_o_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-221.7, 88.3, 283.6)",23,14,37,-221.7,88.3,283.6,14,222.283100
2,P10,1.613037e+12,wealth_6_e2_15_r_o_v1_old,"Checkpoint Pos : (287.9, 63.4, 636.1)","Stimuli Pos : (-116.3, 63.4, 272.2)",4,1,46,-116.3,63.4,272.2,7,543.870500
3,P10,1.613037e+12,pirate_2_e3_11_nr_h_v1_old,"Checkpoint Pos : (-320.1, 88.3, 482.9)","Stimuli Pos : (-38.6, 88.3, 1182.9)",78,100,78,-38.6,88.3,1182.9,1,754.491100
4,P10,1.613037e+12,viking_8_e3_16_nr_h_v1_old,"Checkpoint Pos : (333.1, 53.3, 701.5)","Stimuli Pos : (-142.3, 53.3, 1275.2)",32,100,21,-142.3,53.3,1275.2,1,745.093800
6,P10,1.613037e+12,pirate_7_e3_1_nr_h_v1_old,"Checkpoint Pos : (-128.5, 90.6, 220.8)","Stimuli Pos : (-129.3, 90.6, 220.4)",78,14,82,-129.3,90.6,220.4,1,0.952219


In [24]:
groups = df.groupby('Participant')

save_path = '../data/processed/rsa/judgment_distances/'
os.makedirs(save_path, exist_ok=True)
df.to_csv(save_path + 'all_participants_judgments.csv', index=False, sep=',')

for participant, group in groups:

    ## Spatial Distances

    participant_path = save_path + f'sub-{participant}/'
    os.makedirs(participant_path, exist_ok=True)
    judged_positions = group[['X', 'Y', 'Z']].to_numpy()
    dist_matrix = squareform(pdist(judged_positions))

    plt.figure(figsize=(8, 6))
    plt.imshow(dist_matrix, cmap='viridis', interpolation='nearest')
    plt.colorbar(label='Distance')
    plt.title(f'Judged Distances - Participant {participant}')
    plt.xlabel('Stimulus Index')
    plt.ylabel('Stimulus Index')
    plt.savefig(participant_path + f'sub-{participant}_judged_spatial_distances.png')
    plt.close()

    # save matrix as py
    np.save(participant_path + f'sub-{participant}_judged_spatial_distances.npy', dist_matrix)

    # Plot the dots in 3D
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(group['X'], group['Y'], group['Z'], c='b', marker='o')
    ax.set_title(f'Judged Positions - Participant {participant}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    plt.savefig(participant_path + f'sub-{participant}_judged_positions.png')
    plt.close()

    # percentage of confidence below 50
    low_confidence_percentage = (group['Placed Confidence'].astype(float) < 50).mean() * 100
    if low_confidence_percentage > 90:
        print(f'Participant {participant} has {low_confidence_percentage:.2f}% low spatial confidence ratings & should probably be discarded.')

    ## Temporal Distances
    # Disregard where timed days == 100
    group = group[group['Timed (days)'] != 100]
    # Save as csv file column V4 for further processing
    group[['V4']].to_csv(participant_path + f'sub-{participant}_timed_valid.csv', index=False, sep=',')
    if len(group) < 2:
        print(f'Participant {participant} has less than 2 valid temporal judgments & should probably be discarded.')
        continue
    judged_positions = group[['Timed (days)']].to_numpy()
    dist_matrix = squareform(pdist(judged_positions))

    # save matrix as py
    np.save(participant_path + f'sub-{participant}_judged_temporal_distances.npy', dist_matrix)

    plt.figure(figsize=(8, 6))
    plt.imshow(dist_matrix, cmap='viridis', interpolation='nearest')
    plt.colorbar(label='Distance')
    plt.title(f'Judged Temporal Distances - Participant {participant}')
    plt.xlabel('Stimulus Index')
    plt.ylabel('Stimulus Index')
    plt.savefig(participant_path + f'sub-{participant}_judged_temporal_distances.png')
    plt.close()

    # percentage of confidence below 50
    low_confidence_percentage = (group['Timed Confidence'].astype(float) < 50).mean() * 100
    if low_confidence_percentage > 90:
        print(f'Participant {participant} has {low_confidence_percentage:.2f}% low temporal confidence ratings & should probably be discarded.')


Participant P14 has 95.83% low spatial confidence ratings & should probably be discarded.
Participant P27 has 91.67% low spatial confidence ratings & should probably be discarded.
Participant P34 has 95.83% low spatial confidence ratings & should probably be discarded.
Participant P44 has 97.92% low spatial confidence ratings & should probably be discarded.
Participant P47 has 97.78% low temporal confidence ratings & should probably be discarded.
